## 07.1 — Matching fibroblast subtypes (Fib_1–Fib_6) to the reference cerebrovascular atlas

**Reference:** Wang et al., *Spatial atlas of the human brain vasculature reveals specialized cell ensembles*, Cell (2026) — `Data/mmc9.pdf` (main text + STAR Methods, corrected proof, doi:10.1016/j.cell.2026.07.007), with marker genes cross-checked against `Data/mmc3.xlsx` (Table S2, full per-subtype DE marker tables).

**Goal:** Wang et al. resolved human brain fibroblasts into **5 molecularly and spatially defined subsets** from an integrated sc/snRNA-seq atlas (39 donors) validated with spatial transcriptomics and immunostaining in temporal cortex/hippocampus:

| Paper subtype | Identity marker | Anatomic niche | Reported function (GO/pathway) |
|---|---|---|---|
| Dural border FB | **KCNMA1** | Arachnoid layers / subarachnoid space | Contractile proteins |
| Arachnoid barrier FB | **STXBP6** | Arachnoid layers / subarachnoid space | Adherens junction complexes |
| Inner arachnoid FB | **SLC5A5** (+SLC13A3, SLC22A6) | Arachnoid layers / subarachnoid space | Solute carrier transport (CSF composition) |
| Pial FB | **NGFR** (+FBLN2, LUM) | Pia mater; ensheathes larger arteries/veins | Extracellular matrix (stromal support) |
| Parenchymal perivascular FB | **ABCA8** (+ABCA10, NTRK3) | Penetrating arterioles/venules in gray & white matter (not capillaries) | ABC transporters; cell signaling/synaptic plasticity |

Pan-fibroblast identity markers used throughout the paper: **COL1A1, FBLN1**.

The paper explicitly notes as an open question that *"further studies are required to determine how vascular cell states and spatial organization vary across other brain regions"* — its own atlas and spatial validation are limited to temporal cortex and hippocampus. Our dataset (41 brain regions incl. choroid plexus, leptomeninges, and the 3 major intracranial arteries, 4 donors) is a direct extension into that open question, and Part 1 here first asks the prerequisite question: **do our independently-derived Fib_1–Fib_6 Leiden clusters correspond to these 5 published subtypes at all?**

**Method.** `Data/mmc3.xlsx` (Table S2, sheet "FB") is the paper's full one-vs-rest Wilcoxon marker result for each of its 5 named fibroblast subtypes, so we now use the paper's own top-log2FC markers directly as the reference gene set (see below), rather than a hand-picked shortlist from the main text. For each of our clusters we test: (1) where these reference genes rank among that cluster's own Wilcoxon markers, (2) an `AddModuleScore` signature score per reference subtype, (3) independent GO enrichment of each cluster's top markers compared against the paper's reported themes, and (4) the anatomic-niche enrichment of each cluster (Barrier = choroid plexus + leptomeninges; Major Vessel = MCA/ACA/BA; the rest = parenchymal regions), which should recapitulate the niches in the table above if the identities are conserved.

In [ ]:
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(ggplot2)
    library(schard)
    library(viridis)
    library(RColorBrewer)
    library(ComplexHeatmap)
    library(colorRamp2)
    library(clusterProfiler)
    library(org.Hs.eg.db)
    library(readxl)
})

## working directory -- self-contained within this repo (see CLAUDE.md path caveat)
dir <- "/home/kibr/Work/Fibroblast_regional_difference/"
setwd(dir)
dir.create("./Results/DEG", showWarnings = FALSE, recursive = TRUE)
dir.create("./Results/Tables", showWarnings = FALSE, recursive = TRUE)
dir.create("./Figures", showWarnings = FALSE, recursive = TRUE)

celltype_colors <- c(
  "Fib_1" = "#00695C", "Fib_2" = "#00796B", "Fib_3" = "#009688",
  "Fib_4" = "#26A69A", "Fib_5" = "#4DB6AC", "Fib_6" = "#80CBC4"
)

### Load data and apply the same QC used in `06.4_R_DEG_analysis_fibroblast.ipynb`

In [ ]:
h5ad_file <- "./Data/Fibroblast_temp_processed.h5ad"
obj_oi <- schard::h5ad2seurat(h5ad_file)

## exclude samples with less than 3 cells (consistent with 06.4)
meta <- obj_oi@meta.data
mat <- table(meta$sampleID)
sample_excluded <- names(mat[mat < 3])
cat(length(sample_excluded), "sample(s) excluded for <3 cells\n")
obj_oi <- subset(obj_oi, subset = !sampleID %in% sample_excluded)

obj_oi <- NormalizeData(obj_oi)
Idents(obj_oi) <- obj_oi$Cell_type
obj_oi

table(obj_oi$Cell_type)

### Recompute cluster markers (Wilcoxon, as in `06.4`) so this notebook is self-contained

In [ ]:
all.markers <- FindAllMarkers(obj_oi, verbose = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
write.csv(all.markers, file = "./Results/DEG/Table_S8_Fibroblast_cell_type_markers.csv")
dim(all.markers)

### Reference marker table from Wang et al. 2026 (`Data/mmc3.xlsx`, Table S2, sheet "FB")

Previously this table was a small hand-curated shortlist (13 genes) pulled from the main text.
`Data/mmc3.xlsx` is the paper's actual supplementary Table S2 — the full one-vs-rest Wilcoxon
marker result for each of the paper's 5 named fibroblast subtypes (24,428 gene x subtype rows
in the "FB" sheet: `Gene`, `P value`, `Average log2 fold change`, `Pct.1`, `Pct.2`,
`P value adjusted`, `FB subset`). We now use that directly instead of the hand-picked list.

Two things need handling before it's usable:

1. **Only markers with high log2FC.** Most of the ~1,400-4,800 significant (`padj < 0.05`)
   genes per subtype are weak. We keep only genes that are both significant and reasonably
   specific to that subtype (`pct.1 > 0.25`, mirroring the `min.pct = 0.25` threshold used for
   our own `FindAllMarkers` call above), then take the **top 10 per subtype ranked by log2FC**.
2. **A gene-symbol artifact in the source file.** A subset of gene symbols in the sheet carry a
   stray digit fused onto the end (e.g. `STXBP61` alongside a separately-listed, clean
   `STXBP6`) — almost certainly a footnote-marker character that collapsed into the text on
   export. We detect and fix this by stripping a trailing digit whenever the raw symbol isn't
   one of our own genes but the stripped form is.

Pan-fibroblast identity markers (**COL1A1, FBLN1**) aren't part of Table S2 (they're not
subtype-specific by definition, so they don't show up in a one-vs-rest contrast) and are kept
in by hand as before.

In [ ]:
## Load Table S2, sheet "FB" -- title row + blank row precede the real header (see raw inspection)
mmc3_fb <- read_excel("./Data/mmc3.xlsx", sheet = "FB", skip = 2)
colnames(mmc3_fb) <- c("gene", "p_val", "log2FC", "pct.1", "pct.2", "p_val_adj", "subtype_raw")

## Fix the stray-footnote-digit artifact: strip a trailing digit only when the raw symbol is
## not one of our own genes but the stripped form is (avoids mangling real symbols like SLC9A2)
our_genes <- rownames(obj_oi)
raw_gene <- mmc3_fb$gene
stripped <- substr(raw_gene, 1, nchar(raw_gene) - 1)
needs_fix <- !(raw_gene %in% our_genes) & grepl("[0-9]$", raw_gene) & (stripped %in% our_genes)
cat(sum(needs_fix), "gene symbols fixed (stray trailing digit stripped), e.g.:\n")
print(head(unique(data.frame(raw = raw_gene[needs_fix], fixed = stripped[needs_fix])), 8))
mmc3_fb$gene[needs_fix] <- stripped[needs_fix]

## Paper subtype label -> naming convention already used elsewhere in this notebook
subtype_map <- c(
  "Dural_Border_FB"      = "Dural border FB",
  "Arachnoid_Barrier_FB" = "Arachnoid barrier FB",
  "Inner_Arachnoid_FB"   = "Inner arachnoid FB",
  "Pial_FB"              = "Pial FB",
  "Parenchymal_FB"       = "Parenchymal perivascular FB"
)
mmc3_fb$subtype <- subtype_map[mmc3_fb$subtype_raw]

## Keep markers with HIGH log2FC only: significant (padj<0.05) and reasonably specific
## (pct.1 > 0.25, i.e. detected in >25% of that subtype's own cells), then the top 10 per
## subtype ranked by log2FC -- not just "significant", which would keep thousands of weak hits.
N_TOP <- 10
ref_markers_top <- mmc3_fb %>%
    filter(p_val_adj < 0.05, pct.1 > 0.25) %>%
    group_by(subtype) %>%
    arrange(desc(log2FC)) %>%
    slice_head(n = N_TOP) %>%
    ungroup() %>%
    transmute(subtype, gene, role = "Table S2 top-log2FC marker", log2FC = round(log2FC, 2))

## Pan-fibroblast markers stated in the paper's main text -- not a one-vs-rest DE result, so
## not present in Table S2's per-subtype contrasts; kept in by hand as before.
pan_markers <- tibble(subtype = "Pan-fibroblast", gene = c("COL1A1", "FBLN1"),
                       role = "pan marker (main text)", log2FC = NA_real_)

ref_markers <- bind_rows(ref_markers_top, pan_markers)
write.csv(ref_markers, "./Results/Tables/reference_markers_from_mmc3_top10.csv", row.names = FALSE)
as.data.frame(ref_markers %>% arrange(subtype, desc(log2FC)))

### Dot plot of the reference marker genes across our Fib_1–Fib_6 clusters

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 5, repr.plot.res = 150)

## group features by subtype so the dot plot facets into labeled panels, in the same
## order the subtypes are introduced in the intro table above. Facet labels are shortened
## (vs. the full "... FB" names used elsewhere) so the 2-gene Pan-fibroblast panel is wide
## enough to hold its own strip label without clipping.
subtype_order <- c("Dural border FB", "Arachnoid barrier FB", "Inner arachnoid FB",
                    "Pial FB", "Parenchymal perivascular FB", "Pan-fibroblast")
facet_label   <- c("Dural border", "Arachnoid barrier", "Inner arachnoid",
                    "Pial", "Parenchymal perivascular", "Pan-FB")
feature_list <- split(ref_markers$gene, factor(ref_markers$subtype, levels = subtype_order))
names(feature_list) <- facet_label

p_dot <- DotPlot(obj_oi, features = feature_list, cols = viridis(3, option = "magma"), dot.scale = 5) +
    RotatedAxis() +
    theme(axis.title = element_blank(),
          axis.text.x = element_text(size = 7.5),
          strip.background = element_rect(fill = "grey92", color = NA),
          strip.text = element_text(size = 8.5, face = "bold"),
          panel.spacing = unit(0.6, "lines"),
          plot.title = element_text(size = 11)) +
    ggtitle("Wang et al. 2026 Table S2 fibroblast markers (top 10 by log2FC per subtype, padj<0.05 & pct.1>0.25) across Fib_1-6")
p_dot
ggsave("./Figures/Figure_07_1A_reference_marker_dotplot.pdf", p_dot, width = 16, height = 5)

### Signature (module) score per reference subtype, and per-cluster mean

In [ ]:
subtypes <- unique(ref_markers$subtype[ref_markers$subtype != "Pan-fibroblast"])
gene_sets <- lapply(subtypes, function(s) unique(ref_markers$gene[ref_markers$subtype == s]))
names(gene_sets) <- subtypes

obj_oi <- AddModuleScore(obj_oi, features = gene_sets, name = "RefScore_", search = FALSE)
score_cols <- paste0("RefScore_", seq_along(gene_sets))
colnames(obj_oi@meta.data)[match(score_cols, colnames(obj_oi@meta.data))] <- paste0("Score_", make.names(names(gene_sets)))
score_names <- paste0("Score_", make.names(names(gene_sets)))

mean_score <- obj_oi@meta.data %>%
    group_by(Cell_type) %>%
    summarise(across(all_of(score_names), mean), .groups = "drop")
mat <- as.matrix(mean_score[, score_names]); rownames(mat) <- mean_score$Cell_type
colnames(mat) <- gsub("Score_|\\.FB$", "", colnames(mat))
mat_z <- scale(mat)
round(mat_z, 2)

In [ ]:
options(repr.plot.width = 4, repr.plot.height = 4, repr.plot.res = 300)
pdf("./Figures/Figure_07_1B_reference_score_heatmap.pdf", width = 5, height = 5)
ht <- Heatmap(mat_z, name = "z-score",
            #   col = colorRamp2(c(-2, 0, 2), c("#5387be", "#F7F7F7", "#b31f2c")),
              col = viridis(3, option = "viridis"),
              cluster_rows = FALSE, cluster_columns = FALSE,
              row_names_side = "left", column_names_rot = 35,
              ## only show the z-score value in the cell if it's >0.5, otherwise leave it blank
              cell_fun = function(j, i, x, y, w, h, fill) {
                  if (mat_z[i, j] > 0.5) {
                      grid.text(sprintf("%.1f", mat_z[i, j]), x, y, gp = gpar(fontsize = 11))
                  }
              })
draw(ht)
dev.off()
draw(ht)